# Integración reproducible — Rol 5, Día 1

## Objetivo
Conectar la melodía procesada del silbido con la representación de ocho genes, sintetizar una flauta y evaluar el audio con el fitness implementado. Esta es una prueba de interfaces y reproducibilidad; **no es una corrida del algoritmo genético**.

Las celdas usan datos versionados, no escriben resultados experimentales en archivos y no ejecutan el plan. Los outputs visibles son solo la comprobación de integración.

## Plan experimental documentado (no ejecutado)

La configuración base, los tamaños y las semillas se muestran directamente desde las interfaces de configuración del Rol 4.

`crear_plan_experimental()` genera las 12 configuraciones que se muestran abajo y `SEMILLAS_EXPERIMENTALES` aporta las semillas previstas como réplicas. El plan se deriva del módulo de configuración y no se ejecuta en este notebook. Aunque el código define los factores de la matriz, se debe confirmar que los niveles de elitismo y mutación son los acordados; la métrica fitness oficial también sigue pendiente de confirmación. Cruce y mutación no se inventan ni se ejecutan aquí.

In [15]:
from pathlib import Path
import sys

import numpy as np
import soundfile as sf

RAIZ = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "src").is_dir() and (ruta / "datos").is_dir()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio desde el directorio actual.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.configuracion import (
    SEMILLAS_EXPERIMENTALES,
    TAMANOS_POBLACION_EXPERIMENTALES,
    ConfiguracionAG,
    crear_plan_experimental,
)
from src.algoritmo_genetico import (
    calcular_estadisticas,
    evaluar_poblacion,
    inicializar_poblacion,
    seleccionar_elites,
)
from src.extraccion import cargar_melodia_csv
from src.fitness import evaluar_audio, evaluar_individuo
from src.representacion import (
    GENES_SCHEMA,
    generar_individuo_aleatorio,
    obtener_nombres_genes,
    validar_individuo,
)
from src.operadores import seleccion_torneo
from src.sintesis import FRECUENCIA_MUESTREO, sintetizar_instrumento

print(f"Raíz del proyecto: {RAIZ}")
print("Genes:", obtener_nombres_genes())

config_base = ConfiguracionAG()
plan = crear_plan_experimental()
assert len(plan) == 12
campos_acordados = (
    "tam_poblacion",
    "max_generaciones",
    "elitismo",
    "tam_torneo",
    "paciencia_estancamiento",
    "tolerancia_mejora",
)
print("Parámetros base comunicados por Rol 4:", {
    campo: getattr(config_base, campo) for campo in campos_acordados
})
print("Tamaños previstos:", TAMANOS_POBLACION_EXPERIMENTALES)
print("Semillas previstas (réplicas, no ejecutadas):", SEMILLAS_EXPERIMENTALES)
print("Configuraciones generadas por crear_plan_experimental() (no ejecutadas):")
print("N, elitismo, probabilidad_mutacion (factores a ratificar)")
for indice, configuracion in enumerate(plan, start=1):
    print(
        indice,
        configuracion.tam_poblacion,
        configuracion.elitismo,
        configuracion.probabilidad_mutacion,
    )

Raíz del proyecto: c:\Users\Rodrigo\Desktop\TAREAS_UCB\Ultimo_Semestre\Inteligencia_Artificial\Silbido\ProyectoSilbido\silbido-a-instrumento-prototipo-AG-IA
Genes: ['amp_arm1', 'amp_arm2', 'amp_arm3', 'ataque_s', 'decaimiento_s', 'sostenido_rel', 'relajacion_s', 'ruido_soplo']
Parámetros base comunicados por Rol 4: {'tam_poblacion': 50, 'max_generaciones': 50, 'elitismo': 2, 'tam_torneo': 3, 'paciencia_estancamiento': 10, 'tolerancia_mejora': 0.0001}
Tamaños previstos: (20, 50, 100)
Semillas previstas (réplicas, no ejecutadas): (42, 101, 2024, 7, 999)
Configuraciones generadas por crear_plan_experimental() (no ejecutadas):
N, elitismo, probabilidad_mutacion (factores a ratificar)
1 20 1 0.02
2 20 1 0.08
3 20 2 0.02
4 20 2 0.08
5 50 1 0.02
6 50 1 0.08
7 50 2 0.02
8 50 2 0.08
9 100 1 0.02
10 100 1 0.08
11 100 2 0.02
12 100 2 0.08


## Datos y conexión con extracción

`cargar_melodia_csv` lee la melodía que Rol 1 ya procesó y guardó en CSV; no vuelve a detectar notas desde el silbido. La referencia se carga como audio mono. El fitness exige audios alineados y con igual número de muestras, por lo que esta comprobación falla explícitamente si las entradas no cumplen ese contrato.

In [16]:
RUTA_MELODIA = RAIZ / "datos" / "procesados" / "melodia_01.csv"
RUTA_REFERENCIA = RAIZ / "datos" / "audio" / "referencias" / "flauta_referencia_01.wav"
notas, duraciones = cargar_melodia_csv(RUTA_MELODIA)
referencia, frecuencia_referencia = sf.read(RUTA_REFERENCIA, dtype="float32")

if referencia.ndim != 1:
    raise ValueError("La referencia debe ser mono para evaluar fitness.")
if frecuencia_referencia != FRECUENCIA_MUESTREO:
    raise ValueError(
        f"Frecuencia inesperada: {frecuencia_referencia} Hz; "
        f"se esperaba {FRECUENCIA_MUESTREO} Hz."
    )
fronteras = np.rint(
    np.concatenate(([0.0], np.cumsum(duraciones))) * frecuencia_referencia
).astype(np.int64)
if referencia.size != int(fronteras[-1]):
    raise ValueError(
        "La melodía y la referencia no tienen igual longitud; "
        "no se recortan ni remuestrean implícitamente."
    )
print(
    f"Eventos: {len(notas)} | muestras: {referencia.size} | "
    f"duración: {referencia.size / frecuencia_referencia:.3f} s"
)

Eventos: 16 | muestras: 59645 | duración: 2.705 s


## Prueba inicial de integración (no experimental)

Se genera un cromosoma válido con una semilla fija, se sintetiza el mismo cromosoma dos veces con semilla fija y se comprueba que el audio sea idéntico. Después se evalúan por separado las métricas reales del módulo de fitness. La celda siguiente comprueba además selección por torneo, inicialización, evaluación de población, elitismo y estadísticas usando el audio de referencia real. No se ejecuta un ciclo generacional ni se escriben archivos de resultados; los valores mostrados son salidas de prueba, no resultados experimentales.

In [17]:
config_integracion = ConfiguracionAG(
    tam_poblacion=3,
    elitismo=2,
    semilla=config_base.semilla,
)
rng_integracion = np.random.default_rng(config_integracion.semilla)
poblacion_integracion = inicializar_poblacion(
    config_integracion,
    rng=rng_integracion,
)
if poblacion_integracion.shape != (config_integracion.tam_poblacion, len(GENES_SCHEMA)):
    raise AssertionError("La inicialización no devolvió la forma de población esperada.")

aptitudes_integracion = evaluar_poblacion(
    poblacion_integracion,
    lambda individuo: evaluar_individuo(
        individuo,
        notas,
        duraciones,
        referencia,
        metrica="stft",
    ),
)
if aptitudes_integracion.shape != (len(poblacion_integracion),):
    raise AssertionError("La evaluación no conservó una aptitud por individuo.")

padre = seleccion_torneo(
    poblacion_integracion,
    aptitudes_integracion,
    k=config_integracion.tam_torneo,
    rng=rng_integracion,
)
elites_integracion = seleccionar_elites(
    poblacion_integracion,
    aptitudes_integracion,
    cantidad=config_integracion.elitismo,
)
estadisticas_integracion = calcular_estadisticas(
    poblacion_integracion,
    aptitudes_integracion,
    generacion=0,
)

assert any(np.array_equal(padre, candidato) for candidato in poblacion_integracion)
assert np.array_equal(elites_integracion[0], poblacion_integracion[np.argmax(aptitudes_integracion)])
assert not np.shares_memory(elites_integracion, poblacion_integracion)
assert np.isclose(estadisticas_integracion.fitness_maximo, np.max(aptitudes_integracion))
assert np.isclose(estadisticas_integracion.fitness_medio, np.mean(aptitudes_integracion))
assert np.isfinite(estadisticas_integracion.diversidad_genetica)

print("Población inicial evaluada con fitness real:", len(poblacion_integracion))
print("Fitness en orden población-individuo:", aptitudes_integracion)
print("Padre seleccionado por torneo k=", config_integracion.tam_torneo)
print("Número de élites independientes:", len(elites_integracion))
print("Estadísticas de la población inicial:", estadisticas_integracion)
print("Pruebas de interfaces disponibles completadas; no se ejecutó ciclo generacional.")

Población inicial evaluada con fitness real: 3
Fitness en orden población-individuo: [0.5150123  0.6453372  0.52458545]
Padre seleccionado por torneo k= 3
Número de élites independientes: 2
Estadísticas de la población inicial: EstadisticasGeneracion(generacion=0, fitness_maximo=0.6453372035351678, fitness_medio=0.5616449842387135, desviacion_fitness=0.05930824529128079, diversidad_genetica=0.2325021127489916)
Pruebas de interfaces disponibles completadas; no se ejecutó ciclo generacional.


In [18]:
cromosoma = generar_individuo_aleatorio(
    rng=np.random.default_rng(config_base.semilla)
)
valido, mensaje = validar_individuo(cromosoma)
if not valido:
    raise ValueError(f"El generador produjo un cromosoma inválido: {mensaje}")

SEMILLA_SINTESIS = 2026
audio_prueba = sintetizar_instrumento(
    notas, duraciones, cromosoma, semilla=SEMILLA_SINTESIS
)
audio_repetido = sintetizar_instrumento(
    notas, duraciones, cromosoma, semilla=SEMILLA_SINTESIS
)
if not np.array_equal(audio_prueba, audio_repetido):
    raise AssertionError("La síntesis no fue reproducible con la misma semilla.")
if audio_prueba.ndim != 1 or audio_prueba.size != referencia.size:
    raise AssertionError("La síntesis no coincide con forma/longitud de la referencia.")

metricas = evaluar_audio(audio_prueba, referencia)
fitness_stft_interfaz = evaluar_individuo(
    cromosoma, notas, duraciones, referencia, metrica="stft"
)
if not np.isclose(fitness_stft_interfaz, metricas["fitness_stft"]):
    raise AssertionError("La interfaz de fitness y la evaluación del audio discrepan.")

print("Cromosoma válido:", valido, "| genes:", len(GENES_SCHEMA))
print("Síntesis reproducible:", True, "| muestras:", audio_prueba.size)
print("Métricas de esta comprobación solamente:", metricas)
print("Fitness STFT por interfaz evaluar_individuo:", fitness_stft_interfaz)

Cromosoma válido: True | genes: 8
Síntesis reproducible: True | muestras: 59645
Métricas de esta comprobación solamente: {'error_stft_db': 9.417011843854251, 'fitness_stft': 0.5150123036653107, 'fitness_croma': 0.906467738862723}
Fitness STFT por interfaz evaluar_individuo: 0.5150123036653107


## Dependencias pendientes de otros roles

La integración comprueba representación, lectura de la melodía procesada, síntesis, fitness, selección por torneo, inicialización, evaluación de población, elitismo y estadísticas con datos reales. `src/algoritmo_genetico.py` también ofrece detección de estancamiento, pero no se prueba como parte de este flujo inicial. Todavía no hay un ciclo generacional completo con cruce y mutación, por lo que esta integración no ejecuta el AG ni las 12 configuraciones. Las firmas y decisiones de esos operadores corresponden al equipo y no se inventan aquí.

El contenido de las celdas de plan es configuración prevista, no evidencia de corridas. No se realizan aquí búsqueda aleatoria comparativa, sensibilidad, gráficos de tornado, encuesta ni entregables de días posteriores.